# WISE-Only AGN Variability Manifold + Zeltyn et al. (2024) Projection

This notebook:
1. Rebuilds the WISE-only manifold from **Sample A** (Hemmati et al. 2026, arXiv:2601.08037)
2. Trains a UMAP embedding using GP-interpolated W1 light curves with DTW distance
3. Loads the **Zeltyn et al. (2024)** CL-AGN sample (arXiv:2401.01933, SDSS-V first-year)
4. Projects the Zeltyn targets onto the trained manifold using `mapper.transform()`
5. Visualises whether the new CL-AGNs land near the known turn-on/turn-off region

**Reference configuration from the paper:**
- GP regression (`unify_lc_gp`), RBF kernel `length_scale=200`, `xres=160`
- Single band: **W1** (clearest manifold per Figure 4–7 of the paper)
- Normalise by σ-clipped max of W1
- UMAP: `n_neighbors=50`, `min_dist=0.9`, `metric=dtw_distance`

## 0. Setup

In [ ]:
import sys, os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import astropy.units as u
import umap
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier
from collections import Counter
from matplotlib.colors import LinearSegmentedColormap

NOTEBOOK_DIR = os.path.abspath('')
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, 'code_src'))

from data_structures import MultiIndexDFObject
from wise_functions import wise_get_lightcurves
from sample_selection import clean_sample
from AGNzoo_functions import (
    unify_lc, unify_lc_gp, stat_bands, combine_bands,
    normalize_clipmax_objects, shuffle_datalabel, dtw_distance,
    stretch_small_values_arctan, translate_bitwise_sum_to_labels, update_bitsums,
)

plt.style.use('bmh')
RANDOM_STATE = 20
DATA_DIR = os.path.join(NOTEBOOK_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)
print('Ready. DATA_DIR =', DATA_DIR)

## 1. Download & Load Sample A data

In [ ]:
PARQUET = os.path.join(DATA_DIR, 'df_lc_020724.parquet.gzip')

if not os.path.exists(PARQUET):
    print('Downloading from Google Drive …')
    import subprocess
    subprocess.run(['gdown', '1gb2vWn0V2unstElGTTrHIIWIftHbXJvz', '-O', PARQUET], check=True)
    print('Done.')
else:
    print('File already exists, skipping download.')

# Load exactly as in ML_AGNzoo.md
df_lc = pd.read_parquet(PARQUET)
df_lc = df_lc[df_lc.index.get_level_values('label') != '64']   # remove SPIDER-only (too large)
df_lc = update_bitsums(df_lc)                                   # clean bitwise sums after removing 64

print('Bands in dataset:', df_lc.index.get_level_values('band').unique().tolist())
print('Total objects:   ', df_lc.index.get_level_values('objectid').nunique())

## 2. WISE-only preprocessing (Section 5 of ML_AGNzoo)

In [ ]:
# Following ML_AGNzoo.md Section 5 — GP interpolation on W1+W2, xres=160
bands_inlc = ['W1', 'W2']

objectsw, dobjectsw, flabelsw, keepsw = unify_lc_gp(
    df_lc, bands_inlc, xres=160, numplots=0, low_limit_size=5
)
print(f'Objects after GP cut: {len(objectsw)}')

fvarw, maxarrayw, meanarrayw = stat_bands(objectsw, dobjectsw, bands_inlc, sigmacl=5)
dat_notnormalw = combine_bands(objectsw, bands_inlc)
datw = normalize_clipmax_objects(dat_notnormalw, maxarrayw, band=-1)  # normalise by W2 (last band)

# Drop NaN rows (objects where clipped max = 0)
valid = ~np.isnan(datw).any(axis=1)
if (~valid).sum():
    print(f'Dropping {(~valid).sum()} objects with NaN features')
datw      = datw[valid]
flabelsw  = [l for l, v in zip(flabelsw, valid) if v]
fvarw     = fvarw[:, valid]
maxarrayw = maxarrayw[:, valid]
meanarrayw = meanarrayw[:, valid]

dataw, fzrw, pw = shuffle_datalabel(datw, flabelsw)
fvar_arrw = fvarw[:, pw]

# Build labc using translate_bitwise_sum_to_labels (same as ML_AGNzoo.md)
labcw = {}
for index, f in enumerate(fzrw):
    for label in translate_bitwise_sum_to_labels(int(f)):
        labcw.setdefault(label, []).append(index)

print('Classes:')
for k, v in sorted(labcw.items(), key=lambda x: -len(x[1])):
    print(f'  {k:20s}: {len(v)}')

## 3. Train UMAP on WISE manifold

In [ ]:
# DTW gives cleanest CLAGN separation (Hemmati+2026); manhattan is much faster
USE_DTW = False   # set True for paper-quality result (slow on large sample)

metric      = dtw_distance if USE_DTW else 'manhattan'
metric_name = 'DTW' if USE_DTW else 'Manhattan'

print(f'Training UMAP ({metric_name}) on {dataw.shape[0]} objects …')
mapp = umap.UMAP(
    n_neighbors=50, min_dist=0.9,
    metric=metric, random_state=RANDOM_STATE,
).fit(dataw)

print('Done. Embedding shape:', mapp.embedding_.shape)

---
## 5. Build Zeltyn et al. (2024) Sample

Zeltyn et al. (2024, arXiv:2401.01933) present 113 spectroscopically confirmed
CL-AGNs from SDSS-V BHM first-year data.  We try to retrieve the table from
VizieR automatically; if unavailable, a CSV fallback is provided.

In [ ]:
def sdss_jname_to_skycoord(jname: str) -> SkyCoord | None:
    """
    Parse an SDSS J-name (e.g. 'SDSS J002806.86-052332.8' or 'J002806.86-052332.8') to SkyCoord.
    """
    # Strip optional 'SDSS ' prefix and the mandatory 'J'
    j = re.sub(r'^(SDSS\s*)?J', '', jname.strip())
    m = re.match(r'(\d{2})(\d{2})(\d{2}\.\d+)([+-])(\d{2})(\d{2})(\d{2}\.\d+)$', j)
    if m is None:
        warnings.warn(f'Could not parse J-name: {jname}')
        return None
    ra_str  = f"{m.group(1)}:{m.group(2)}:{m.group(3)}"
    dec_str = f"{m.group(4)}{m.group(5)}:{m.group(6)}:{m.group(7)}"
    return SkyCoord(ra_str, dec_str, unit=(u.hourangle, u.deg))


def build_zeltyn_sample_table(zeltyn_df: pd.DataFrame) -> Table:
    """
    Convert a DataFrame with columns ['Name', 'Class', 'z'] (and optionally 'RA'/'Dec')
    into an astropy sample_table compatible with wise_get_lightcurves().

    'Class' should be 'CL-AGN' or 'EVQ' (we keep only CL-AGNs by default).
    """
    # Keep confirmed CL-AGNs (both core and RM subsets)
    cl = zeltyn_df[zeltyn_df['Class'].str.startswith('CL-AGN')].copy()

    coords_z, labels_z = [], []
    for _, row in cl.iterrows():
        # Prefer explicit RA/Dec columns if present
        if 'RA' in row and 'Dec' in row and pd.notna(row['RA']):
            c = SkyCoord(ra=row['RA'] * u.deg, dec=row['Dec'] * u.deg)
        else:
            c = sdss_jname_to_skycoord(row['Name'])
        if c is not None:
            coords_z.append(c)
            labels_z.append('Zeltyn24')

    tbl = clean_sample(coords_z, labels_z, consolidate_nearby_objects=False)
    print(f'Zeltyn CL-AGN sample size: {len(tbl)}')
    return tbl

In [ ]:
print(f'Training UMAP (DTW) on {dataw.shape[0]} objects …')
print('This will take ~30–60 min for 1000+ objects — exact paper configuration.')

mapp = umap.UMAP(
    n_neighbors=50, min_dist=0.9,
    metric=dtw_distance, random_state=RANDOM_STATE,
).fit(dataw)

metric_name = 'DTW'
print('Done. Embedding shape:', mapp.embedding_.shape)

In [ ]:
# ── CSV fallback ──────────────────────────────────────────────────────────────
# If VizieR is unavailable, create a CSV file at the path below with columns:
#   Name        – SDSS J-name  (e.g.  SDSS J002806.86-052332.8)
#   Class       – 'CL-AGN' or 'EVQ'
#   z           – spectroscopic redshift
# Optionally add columns RA and Dec (decimal degrees) to skip J-name parsing.

ZELTYN_CSV = os.path.join(DATA_DIR, 'zeltyn2024_table5.csv')

if zeltyn_df is None:
    if os.path.exists(ZELTYN_CSV):
        zeltyn_df = pd.read_csv(ZELTYN_CSV)
        print(f'Loaded {len(zeltyn_df)} rows from {ZELTYN_CSV}')
    else:
        raise FileNotFoundError(
            f'Neither VizieR nor the CSV fallback ({ZELTYN_CSV}) is available.\n'
            'Please download Table 5 from https://arxiv.org/abs/2401.01933 '
            'and save it as a CSV with columns: Name, Class, z.'
        )

# Inspect
print(zeltyn_df.head())
print('Class counts:', zeltyn_df['Class'].value_counts().to_dict())

In [ ]:
# Build the sample_table for Zeltyn CL-AGNs
sample_table_Z = build_zeltyn_sample_table(zeltyn_df)

---
## 6. Fetch WISE Light Curves for Zeltyn Targets

In [ ]:
ZELTYN_PARQUET = os.path.join(DATA_DIR, 'df_lc_zeltyn_wise.parquet.gzip')

if os.path.exists(ZELTYN_PARQUET):
    print('Loading cached Zeltyn light curves …')
    df_lc_Z = MultiIndexDFObject(data=pd.read_parquet(ZELTYN_PARQUET))
else:
    print('Fetching WISE light curves for Zeltyn targets …')
    df_lc_Z = wise_get_lightcurves(
        sample_table_Z,
        radius=1.0,
        bandlist=['WISE_W1', 'WISE_W2'],
    )
    df_lc_Z.data.to_parquet(ZELTYN_PARQUET, compression='gzip')
    print(f'Saved to {ZELTYN_PARQUET}')

# Rename bands (same as Section 2.1)
df_lc_Z.data.index = df_lc_Z.data.index.set_levels(
    df_lc_Z.data.index.levels[df_lc_Z.data.index.names.index('band')].str.replace('WISE_', '', regex=False),
    level='band'
)

n_with_data = df_lc_Z.data.index.get_level_values('objectid').nunique()
print(f'Zeltyn objects with WISE data: {n_with_data} / {len(sample_table_Z)}')

---
## 7. Preprocess Zeltyn Light Curves

**Identical** pipeline to Section 3 so the feature vectors are comparable.

In [ ]:
# Identical pipeline to Sample A — W1+W2, GP, same normalisation band
objects_Z, dobjects_Z, flabels_Z, keeps_Z = unify_lc_gp(
    df_lc_Z.data, bands_inlc, xres=160, numplots=0, low_limit_size=5
)
print(f'Zeltyn objects surviving GP cut: {len(objects_Z)}')

fvar_Z, maxarray_Z, meanarray_Z = stat_bands(objects_Z, dobjects_Z, bands_inlc, sigmacl=5)
dat_notnormal_Z = combine_bands(objects_Z, bands_inlc)
dat_Z = normalize_clipmax_objects(dat_notnormal_Z, maxarray_Z, band=-1)

print(f'Zeltyn feature matrix: {dat_Z.shape}')

---
## 8. Project Zeltyn Targets onto the Trained Manifold

`mapper.transform()` performs **out-of-sample** embedding — no retraining.

In [ ]:
embedding_Z = mapp.transform(dat_Z)
print(f'Zeltyn projected: {embedding_Z.shape}')

---
## 9. Final Plot – Zeltyn CL-AGNs on the Sample A Manifold

In [ ]:
# ── Probability density maps (ML_AGNzoo.md Section 5 style) ──────────────────
emb = mapp.embedding_
NBINS = 12
hist_all, x_edges, y_edges = np.histogram2d(emb[:, 0], emb[:, 1], bins=NBINS)

turn_on_idx  = labcw.get('Turn-on',  [])
turn_off_idx = labcw.get('Turn-off', [])
print(f'Turn-on:  {len(turn_on_idx)} | Turn-off: {len(turn_off_idx)} | Zeltyn: {len(embedding_Z)}')

def class_prob_map(indices):
    h, _, _ = np.histogram2d(emb[indices, 0], emb[indices, 1], bins=(x_edges, y_edges))
    with np.errstate(invalid='ignore'):
        return np.where(hist_all > 0, h / hist_all, np.nan)

xc = 0.5 * (x_edges[:-1] + x_edges[1:])
yc = 0.5 * (y_edges[:-1] + y_edges[1:])
XX, YY = np.meshgrid(xc, yc)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, idx, title in zip(axes,
                           [turn_on_idx, turn_off_idx],
                           ['Turn-on', 'Turn-off']):
    cf = ax.contourf(XX, YY, class_prob_map(idx).T, levels=20, cmap='RdYlBu_r', alpha=0.9)
    plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    ax.scatter(emb[:, 0], emb[:, 1], s=5, c='lightgrey', alpha=0.25, zorder=0)
    ax.scatter(embedding_Z[:, 0], embedding_Z[:, 1],
               s=160, marker='*', c='dodgerblue', edgecolors='navy',
               linewidths=0.6, alpha=0.95, zorder=5,
               label=f'Zeltyn+2024 (n={len(embedding_Z)})')
    ax.set_title(title, size=15, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right', framealpha=0.8)
    ax.axis('off')

plt.suptitle(f'WISE W1+W2 UMAP ({metric_name}) — probability density\nStars = Zeltyn+2024 CL-AGNs', size=13)
plt.tight_layout()
out = os.path.join(DATA_DIR, 'zeltyn_overlay.pdf')
plt.savefig(out, bbox_inches='tight', dpi=150)
plt.show()
print('Saved to', out)

## 10. CLAGN Enrichment Map (Proposal Figure)

In [ ]:
from scipy.stats import gaussian_kde
from matplotlib.colors import Normalize

emb = mapp.embedding_
NBINS = 12

# ── CLAGN enrichment: P(CLAGN | location) ────────────────────────────────────
clagn_idx  = labcw.get('Turn-on', []) + labcw.get('Turn-off', [])
sdss_idx   = labcw.get('SDSS_QSO', [])

hist_all,  xe, ye = np.histogram2d(emb[:, 0], emb[:, 1], bins=NBINS)
hist_clagn, _,  _ = np.histogram2d(emb[clagn_idx, 0], emb[clagn_idx, 1], bins=(xe, ye))

with np.errstate(invalid='ignore'):
    enrichment_map = np.where(hist_all > 0, hist_clagn / hist_all, np.nan)

xc = 0.5 * (xe[:-1] + xe[1:])
yc = 0.5 * (ye[:-1] + ye[1:])
XX, YY = np.meshgrid(xc, yc)

# ── Per-Zeltyn-target local enrichment ───────────────────────────────────────
def local_enrichment(point, xe, ye, enrichment_map):
    xi = np.searchsorted(xe[1:], point[0])
    yi = np.searchsorted(ye[1:], point[1])
    xi = np.clip(xi, 0, enrichment_map.shape[0]-1)
    yi = np.clip(yi, 0, enrichment_map.shape[1]-1)
    return enrichment_map[xi, yi]

zeltyn_enrichment = np.array([local_enrichment(p, xe, ye, enrichment_map) for p in embedding_Z])
overall_clagn_fraction = len(clagn_idx) / len(emb)
median_enrichment_factor = np.nanmedian(zeltyn_enrichment) / overall_clagn_fraction

print(f'Overall CLAGN fraction in Sample A:    {100*overall_clagn_fraction:.1f}%')
print(f'Median CLAGN fraction at Zeltyn locs:  {100*np.nanmedian(zeltyn_enrichment):.1f}%')
print(f'Enrichment factor:                     {median_enrichment_factor:.1f}x')

# ── SDSS QSO density contour (background population) ─────────────────────────
kde_sdss = gaussian_kde(emb[sdss_idx].T, bw_method=0.3)
positions = np.vstack([XX.ravel(), YY.ravel()])
sdss_density = kde_sdss(positions).reshape(XX.shape)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

# Enrichment map background
cf = ax.contourf(XX, YY, enrichment_map.T, levels=15, cmap='YlOrRd', alpha=0.85, zorder=1)
cbar = plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('P(CLAGN | location)', size=11)

# SDSS QSO population contour (shows where normal AGN live)
ax.contour(XX, YY, sdss_density.T, levels=5,
           colors='steelblue', linewidths=1.2, alpha=0.6, zorder=2, linestyles='--')

# Zeltyn targets colored by their local enrichment value
sc = ax.scatter(
    embedding_Z[:, 0], embedding_Z[:, 1],
    c=zeltyn_enrichment, cmap='YlOrRd', norm=Normalize(vmin=0, vmax=enrichment_map[~np.isnan(enrichment_map)].max()),
    s=200, marker='*', edgecolors='black', linewidths=0.7, zorder=5,
)

ax.set_title(
    f'WISE UMAP – CLAGN enrichment map
'
    f'Dashed = SDSS QSO population  |  Stars = Zeltyn+2024 CL-AGNs
'
    f'Median enrichment at Zeltyn locations: {median_enrichment_factor:.1f}× field average',
    size=11,
)
ax.axis('off')
plt.tight_layout()
out = os.path.join(DATA_DIR, 'enrichment_zeltyn.pdf')
plt.savefig(out, bbox_inches='tight', dpi=150)
plt.show()
print('Saved to', out)


---
## 10. Save Embeddings

Save the 2-D coordinates for use in the proposal figures.

In [ ]:
pd.DataFrame({
    'umap_x': mapp.embedding_[:, 0],
    'umap_y': mapp.embedding_[:, 1],
    'label':  fzrw,
}).to_csv(os.path.join(DATA_DIR, 'sampleA_embedding.csv'), index=False)

pd.DataFrame({
    'umap_x': embedding_Z[:, 0],
    'umap_y': embedding_Z[:, 1],
    'label':  flabels_Z,
}).to_csv(os.path.join(DATA_DIR, 'zeltyn_embedding.csv'), index=False)

print('Saved embeddings to', DATA_DIR)

---
## Appendix – Notes

### Band naming
- `wise_get_lightcurves()` returns bands named `'WISE_W1'` / `'WISE_W2'`.
- `unify_lc_gp()` internally checks `band == 'W1'` to apply the WISE GP kernel
  (`RBF(length_scale=200)`) and time grid (`linspace(0, 4000, xres)`).
- The renaming in Section 2.1 / 6 bridges this gap.

### Reproducibility
- Set `RANDOM_STATE` at the top; UMAP with DTW is deterministic given the same seed.
- The parquet caches in `../data/` prevent redundant S3 downloads.

### Extending to W2
- Change `BANDS = ['W1']` to `BANDS = ['W1', 'W2']`.
- The paper (Hemmati+2026) reports W1-only gives the cleanest CLAGN separation,
  but W1+W2 carries more information for the full population.

### Zeltyn table
- If VizieR lookup fails, download Table 5 from the published paper
  (arXiv:2401.01933) and save as `../data/zeltyn2024_table5.csv` with columns
  `Name`, `Class`, `z`.  Optionally add `RA` and `Dec` (decimal degrees) to
  skip J-name coordinate parsing.